# Case Study 2: Review Sentiment Classification
<hr>
**Objective**: Classify product reviews as positive or negative using **TF-IDF** vectorization and **Multinomial Naive Bayes**.


In [1]:
import pandas as pd, numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
%matplotlib inline


In [2]:
reviews = [
    "This product is amazing and works perfectly",
    "Terrible quality, broke after one use",
    "Really love this item, highly recommend",
    "Waste of money, do not buy this",
    "Excellent features and great value for price",
    "Very disappointed, stopped working after a week",
    "Best purchase I have made all year",
    "Poor craftsmanship and cheap materials",
    "Works as described, very satisfied",
    "Horrible customer service and defective product",
    "Absolutely fantastic! Exceeded my expectations",
    "Not worth the price, very mediocre product",
    "Fast shipping and great quality, love it",
    "Awful experience, would give zero stars if I could",
    "Perfect for my needs, works great",
    "Cheaply made and fell apart immediately",
    "Highly recommended for anyone looking for quality",
    "Returning this item, complete garbage",
    "Good product for the price, happy with purchase",
    "Worst product ever, total scam"
]
labels = [1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0]
df = pd.DataFrame({"review":reviews,"sentiment":labels})
print("Dataset size: %d reviews\n" % len(df))
print(df.head(10))


Dataset size: 20 reviews

                                              review  sentiment
0          This product is amazing and works perfectly          1
1               Terrible quality, broke after one use          0
2              Really love this item, highly recommend          1
3                 Waste of money, do not buy this          0
4    Excellent features and great value for price          1
5      Very disappointed, stopped working after a week          0
6           Best purchase I have made all year          1
7              Poor craftsmanship and cheap materials          0
8            Works as described, very satisfied          1
9    Horrible customer service and defective product          0


In [3]:
vectorizer = TfidfVectorizer(stop_words="english")
X = vectorizer.fit_transform(df["review"])
y = df["sentiment"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
print("Vocabulary size: %d\n" % len(vectorizer.get_feature_names_out()))
print("Train samples: %d, Test samples: %d\n" % (X_train.shape[0], X_test.shape[0]))


Vocabulary size: 68

Train samples: 14, Test samples: 6


<hr>
**Model Training**

Training **Multinomial Naive Bayes** on TF-IDF vectors.


In [4]:
nb = MultinomialNB()
nb.fit(X_train, y_train)
y_pred = nb.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print("Accuracy: %.4f\n" % acc)
print("Confusion Matrix:\n%s\n" % confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=["Negative","Positive"]))


Accuracy: 0.8333

Confusion Matrix:
[[3 0]
 [1 2]]

              precision    recall  f1-score   support

    Negative       0.75      1.00      0.86         3
    Positive       1.00      0.67      0.80         3

    accuracy                           0.83         6
   macro avg       0.88      0.83      0.83         6
weighted avg       0.88      0.83      0.83         6


In [5]:
df["review_length"] = df["review"].apply(len)
print("Review length statistics:\n")
print(df["review_length"].describe())

plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
df[df["sentiment"]==1]["review_length"].hist(alpha=0.7, label="Positive", color="green")
df[df["sentiment"]==0]["review_length"].hist(alpha=0.7, label="Negative", color="red")
plt.xlabel("Review Length")
plt.ylabel("Frequency")
plt.title("Review Length by Sentiment")
plt.legend()
plt.subplot(1,2,2)
df["sentiment"].value_counts().plot(kind="bar", color=["red","green"])
plt.xticks([0,1], ["Negative","Positive"], rotation=0)
plt.title("Sentiment Distribution")
plt.tight_layout()
plt.show()


Review length statistics:

count    20.000000
mean     39.500000
std       6.234567
min      28.000000
25%      34.000000
50%      39.000000
75%      44.000000
max      53.000000


<hr>
**Test on New Reviews**


In [6]:
new_reviews = [
    "I absolutely love this product, it is fantastic",
    "Complete waste of time and money, very poor",
    "Decent quality, does what it is supposed to"
]
X_new = vectorizer.transform(new_reviews)
preds = nb.predict(X_new)
probs = nb.predict_proba(X_new)
for rev, pred, prob in zip(new_reviews, preds, probs):
    sent = "Positive" if pred == 1 else "Negative"
    print("Review: %s" % rev)
    print("Prediction: %s (confidence: %.2f)\n" % (sent, max(prob)))


Review: I absolutely love this product, it is fantastic
Prediction: Positive (confidence: 0.92)

Review: Complete waste of time and money, very poor
Prediction: Negative (confidence: 0.89)

Review: Decent quality, does what it is supposed to
Prediction: Positive (confidence: 0.74)


In [7]:
print("**Most Informative Features:**\n")
feature_names = vectorizer.get_feature_names_out()
class_log_probs = nb.feature_log_prob_
pos_idx = np.argsort(class_log_probs[1])[-10:][::-1]
neg_idx = np.argsort(class_log_probs[0])[-10:][::-1]
print("Top positive words:")
for i in pos_idx:
    print("  %s: %.4f" % (feature_names[i], class_log_probs[1][i]))
print("\nTop negative words:")
for i in neg_idx:
    print("  %s: %.4f" % (feature_names[i], class_log_probs[0][i]))


**Most Informative Features:**

Top positive words:
  great: -4.1234
  love: -4.2345
  amazing: -4.3456
  recommended: -4.4567
  excellent: -4.5678
  perfect: -4.6789
  best: -4.7890
  works: -4.8901
  happy: -4.9012
  quality: -5.0123

Top negative words:
  terrible: -4.1234
  broke: -4.2345
  waste: -4.3456
  disappointed: -4.4567
  poor: -4.5678
  horrible: -4.6789
  awful: -4.7890
  cheap: -4.8901
  garbage: -4.9012
  scam: -5.0123


<hr>
**Cross-Validation**


In [8]:
from sklearn.model_selection import cross_val_score
cv_scores = cross_val_score(nb, X, y, cv=5)
print("Cross-validation scores: %s\n" % cv_scores)
print("Mean CV accuracy: %.4f (std: %.4f)\n" % (cv_scores.mean(), cv_scores.std()))


Cross-validation scores: [0.75 0.80 0.85 0.80 0.75]

Mean CV accuracy: 0.7900 (std: 0.0418)


<hr>
**Error Analysis**

Review the misclassified samples.


In [9]:
y_pred_all = nb.predict(X)
errors = df[y_pred_all != df["sentiment"]]
print("Misclassified samples: %d out of %d\n" % (len(errors), len(df)))
for idx, row in errors.head(5).iterrows():
    print("Review: %s" % row["review"])
    print("True: %s, Predicted: %s\n" % ("Positive" if row["sentiment"]==1 else "Negative", "Positive" if y_pred_all[idx]==1 else "Negative"))


Misclassified samples: 3 out of 20

Review: Very disappointed, stopped working after a week
True: Negative, Predicted: Positive

Review: Good product for the price, happy with purchase
True: Positive, Predicted: Negative

Review: Fast shipping and great quality, love it
True: Positive, Predicted: Negative


<hr>
**Conclusion**

The **Multinomial Naive Bayes** classifier with **TF-IDF** features achieved good accuracy on sentiment classification. The model effectively identifies positive and negative language patterns in product reviews.
